# eWaterCycle containerized model

This notebook is to showcase/test the containerized model.

We import the LeakyBucket model:

In [2]:
from src.ewatercycle_swmm.model import SWMM, SWMMLocal

NameError: name 'import_bmi' is not defined

To be able to initialize the model, we need forcing data:

In [ ]:
from ewatercycle.base.parameter_set import ParameterSet
import ewatercycle
from pathlib import Path
import numpy as np

swmm_dir = Path.home() / "swmm"
swmm_dir.mkdir(parents=True, exist_ok=True)

input_file = swmm_dir / "example.inp"

parameters = ParameterSet(
    name="SWMM_parameter_files",
    directory=swmm_dir,
    config=Path(input_file),
    target_model='swmm',
)

Now we can start the model. The `LeakyBucket` class uses a container hosted on the Github container registry.

Alternatively, you can start the model using a local container. For this you would modify the command as such:

```py
from ewatercycle.container import ContainerImage
LeakyBucket(forcing=forcing, bmi_image=ContainerImage("local_image:latest"))
```

In [ ]:
model = SWMMLocal(parameter_set=parameters)
cfg_file, _ = model.setup()

In [ ]:
model.initialize(cfg_file)

In [ ]:
model.bmi.get_component_name()

To test the model, we'll make a hydrograph:

In [ ]:
n_sc    = model.get_grid_size(0)
n_nodes = model.get_grid_size(1)
n_links = model.get_grid_size(2)

times       = []
sc_runoff   = []
node_depth  = []
node_flood  = []
link_flow   = []

while model.time < model.end_time:
    times.append(model.get_current_time())
    sc_runoff.append( model.get_value("subcatchment_runoff", np.empty(n_sc)).copy()    )
    node_depth.append(model.get_value("node_depth",          np.empty(n_nodes)).copy() )
    node_flood.append(model.get_value("node_flooding",       np.empty(n_nodes)).copy() )
    link_flow.append( model.get_value("link_flow",           np.empty(n_links)).copy() 
    model.update()

# Convert to arrays and a pandas time axis
hr         = pd.to_datetime([datetime.datetime.fromtimestamp(t, tz=datetime.timezone.utc) for t in times])
sc_runoff  = np.array(sc_runoff)   # shape: (n_steps, n_sc)
node_depth = np.array(node_depth)  # shape: (n_steps, n_nodes)
node_flood = np.array(node_flood)
link_flow  = np.array(link_flow)   # shape: (n_steps, n_links)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("SWMM BMI Wrapper — Example Simulation (6-hour triangular storm)", fontsize=13)

sc_labels   = [f"S{i+1}" for i in range(n_sc)]
node_labels = [f"Node {i+1}" for i in range(n_nodes)]
link_labels = [f"C{i+1}" for i in range(n_links)]

# Subcatchment runoff  [m3/s → L/s for readability]
ax = axes[0, 0]
for i, lbl in enumerate(sc_labels):
    ax.plot(hr, sc_runoff[:, i] * 1000, label=lbl)
ax.set_ylabel("Runoff [L/s]")
ax.set_title("Subcatchment Runoff")
ax.legend()

# Node water depth
ax = axes[0, 1]
for i, lbl in enumerate(node_labels):
    ax.plot(hr, node_depth[:, i], label=lbl)
ax.set_ylabel("Depth [m]")
ax.set_title("Node Water Depth")
ax.legend()

# Node flooding
ax = axes[1, 0]
for i, lbl in enumerate(node_labels):
    ax.plot(hr, node_flood[:, i] * 1000, label=lbl)
ax.set_ylabel("Flooding [L/s]")
ax.set_title("Node Flooding")
ax.legend()

# Link flow
ax = axes[1, 1]
for i, lbl in enumerate(link_labels):
    ax.plot(hr, link_flow[:, i] * 1000, label=lbl)
ax.set_ylabel("Flow [L/s]")
ax.set_title("Conduit Flow")
ax.legend()

for ax in axes.flat:
    ax.set_xlabel("Time")
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
# plt.savefig("swmm_bmi_results.png", dpi=100, bbox_inches="tight")
plt.show()

To make sure that the container running the model is nicely shut down and doesn't keep running in the background (taking up resources) we need to 'finalize' it.

In [ ]:
model.finalize()